# Comparisons

This notebook runs the comparison pipeline (`src/pipeline.py`). For every protein, each pair of its
structures is compared: the experimental structures with each other, each prediction with each
experimental structure, the two predictions with each other, and the representative model of the NMR
ensemble with the remaining models. Every comparison produces one record holding the three Machaon
metrics, the Cα RMSD and the metadata of both structures.

![Pipeline](../docs/pipeline.png)

Each pair is processed in three steps. The two chains are first aligned globally (`src/alignment.py`);
the alignment describes the pair (matched residues, coverage, sequence identity) and supplies the residue
pairs for RMSD. The metadata of both structures is then read (`src/metadata.py`). Finally, each structure
is reduced to a summary of its whole chain and the Machaon metrics compare the two summaries
(`src/metrics.py`); RMSD is added when the paired residues are at least 50 % identical. A pair is
rejected only when a structure cannot be read.

| Category | Pairs | Role |
|---|---|---|
| `exp_vs_exp` | X-ray–NMR, X-ray–cryo-EM, NMR–cryo-EM | experimental variability of the protein |
| `exp_vs_af2`, `exp_vs_of3` | each experimental structure against each prediction | prediction against experiment |
| `af2_vs_of3` | AlphaFold2 against OpenFold3 | agreement between the predictions |
| `nmr_intra_ensemble` | representative NMR model against the other models | spread within one NMR ensemble |

**Input:** `proteins.csv` and `data/`.  
**Output:** `results/comparisons.json` (one record per pair), `results/analysis.json`,
`results/nmr_models.csv` and the run summaries; `python scripts/verify_run.py results` checks them.

In [1]:
import os
import sys

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, ".")

import pandas as pd
from IPython.display import display

from src.pipeline import run_pipeline

## 1. Run the pipeline

Every pair is described first (the alignment is used only for RMSD's residue pairs and for the
coverage and identity columns), then measured. Coverage and identity problems are reported as
warnings in the record, not used to drop the pair: the Machaon metrics are defined for any two
chains, and what differences in coverage do to them is measured in notebook 03.

In [2]:
results = run_pipeline(
    proteins_csv="proteins.csv",
    data_dir="data",
    output_dir="results",
    verbose=True,
    include_nmr_ensemble=True,
    nmr_model_policy="medoid",  # the most central model of each NMR ensemble (src/nmr.py)
)


Protein: Type IV pilin PilE1  (A0A0B7GU52)
  X-ray:   None
  NMR:     data/experimental/6I2O.cif  (representative model: 0)
  Cryo-EM: data/experimental/8PFB.cif
  AF2:     data/alphafold2/AF2_A0A0B7GU52.cif
  OF3:     data/openfold3/OF3_A0A0B7GU52.cif

  Comparing: Type IV pilin PilE1 | NMR vs Cryo-EM
  [ok] [Type IV pilin PilE1 | NMR vs Cryo-EM] 105 matched residues, coverage A/B 92.9%/80.2%, identity 99.0%, RMSD yes

  Comparing: Type IV pilin PilE1 | NMR vs AF2
  [ok] [Type IV pilin PilE1 | NMR vs AF2] 113 matched residues, coverage A/B 100.0%/72.0%, identity 99.1%, RMSD yes
  [warn] [Type IV pilin PilE1 | NMR vs AF2] Low coverage: 113 matched residues (A: 113 residues, coverage_a=100.0%; B: 157 residues, coverage_b=72.0%).  The two chains cover different parts of the protein; whole-chain Machaon metrics include this difference.

  Comparing: Type IV pilin PilE1 | Cryo-EM vs AF2
  [ok] [Type IV pilin PilE1 | Cryo-EM vs AF2] 131 matched residues, coverage A/B 100.0%/83.4%, identity

In [3]:
df = pd.DataFrame(results["comparisons"])
print(f"Total comparisons: {len(df)}")
print(f"Rejected: {len(results['rejected'])}")
df[
    [
        "protein_name",
        "comparison",
        "category",
        "n_matched",
        "coverage",
        "seq_identity",
        "rmsd",
        "t_alpha",
        "w_rdist_norm",
        "b_phipsi",
    ]
].round(4)

Total comparisons: 3477
Rejected: 0


,protein_name,comparison,category,n_matched,coverage,seq_identity,rmsd,t_alpha,w_rdist_norm,b_phipsi
0,Type IV pilin PilE1,NMR_vs_Cryo-EM,exp_vs_exp,105,0.9292,0.9905,4.2054,0.0837,1.1568,0.0269
1,Type IV pilin PilE1,NMR_vs_AF2,exp_vs_af2,113,1.0000,0.9912,3.4251,0.0587,1.2357,0.0670
2,Type IV pilin PilE1,Cryo-EM_vs_AF2,exp_vs_af2,131,1.0000,1.0000,6.5898,0.0236,0.5865,0.0186
3,Type IV pilin PilE1,NMR_vs_OF3,exp_vs_of3,113,1.0000,0.9912,3.6475,0.0609,1.1788,0.0474
4,Type IV pilin PilE1,Cryo-EM_vs_OF3,exp_vs_of3,131,1.0000,1.0000,7.0890,0.0215,0.4564,0.0147
...,...,...,...,...,...,...,...,...,...,...
3472,Phage protein,nmr_intra_model3_vs_model15,nmr_intra_ensemble,130,1.0000,1.0000,0.5902,0.1136,0.0629,0.0034
3473,Phage protein,nmr_intra_model3_vs_model16,nmr_intra_ensemble,130,1.0000,1.0000,0.8316,0.0994,0.0250,0.0000
3474,Phage protein,nmr_intra_model3_vs_model17,nmr_intra_ensemble,130,1.0000,1.0000,0.6570,0.1029,0.0387,0.0003
3475,Phage protein,nmr_intra_model3_vs_model18,nmr_intra_ensemble,130,1.0000,1.0000,0.7032,0.0941,0.0251,0.0003


## 2. Metadata

Every record carries the metadata of both structures. The later notebooks need the entry title and
dates of every experimental structure (flags, training cutoffs), the number of polymer chains in each
entry (context), and the pLDDT and MSA depth of every prediction.

In [4]:
meta_cols = [
    "protein_name",
    "comparison",
    "method_a",
    "resolution_a",
    "n_models_a",
    "method_b",
    "resolution_b",
    "plddt_mean_b",
]
df[meta_cols]

,protein_name,comparison,method_a,resolution_a,n_models_a,method_b,resolution_b,plddt_mean_b
0,Type IV pilin PilE1,NMR_vs_Cryo-EM,NMR,NaN,10,Cryo-EM,3.67,NaN
1,Type IV pilin PilE1,NMR_vs_AF2,NMR,NaN,10,AF2,NaN,78.256561
2,Type IV pilin PilE1,Cryo-EM_vs_AF2,Cryo-EM,3.67,1,AF2,NaN,78.256561
3,Type IV pilin PilE1,NMR_vs_OF3,NMR,NaN,10,OF3,NaN,68.876979
4,Type IV pilin PilE1,Cryo-EM_vs_OF3,Cryo-EM,3.67,1,OF3,NaN,68.876979
...,...,...,...,...,...,...,...,...
3472,Phage protein,nmr_intra_model3_vs_model15,NMR,NaN,20,NMR,NaN,NaN
3473,Phage protein,nmr_intra_model3_vs_model16,NMR,NaN,20,NMR,NaN,NaN
3474,Phage protein,nmr_intra_model3_vs_model17,NMR,NaN,20,NMR,NaN,NaN
3475,Phage protein,nmr_intra_model3_vs_model18,NMR,NaN,20,NMR,NaN,NaN


In [5]:
structures = pd.DataFrame(results["analysis"]["experimental_structures"])
print(f"Experimental structures: {len(structures)}")
for column in ("title", "release_date", "deposition_date"):
    print(f"  without {column}: {int((structures[column] == '').sum())}")

of3_models = df[df["method_b"] == "OF3"].drop_duplicates("protein_name")
missing = int(of3_models["msa_depth_b"].isna().sum())
print(f"OF3 models without MSA depth: {missing} of {len(of3_models)}")

Experimental structures: 392
  without title: 0
  without release_date: 0
  without deposition_date: 0
OF3 models without MSA depth: 0 of 151


## 3. What each number measures

| Number | Computed from | Pairs residues? | Reads as |
|---|---|---|---|
| **w-rdist** | each chain's distribution of all Cα–Cα distances; 1-Wasserstein distance between the two distributions, reported as log10(W + 1) | no | difference in overall size and shape |
| **b-phipsi** | each chain's backbone (φ, ψ) angles in degrees, fitted as a 2-D Gaussian (mean, covariance); Bhattacharyya distance between the two Gaussians | no | difference in secondary-structure make-up |
| **t-alpha** | each chain's heavy atoms, each axis scaled to [0, 1], alpha shape with α = 0.085; exp(\|Δ log n\|) − 1 of the two triangle counts, i.e. n_max / n_min − 1 | no | difference in surface complexity — one number per structure |
| **RMSD** | Cα atoms of the residues paired by the global alignment, after optimal superposition (Kabsch) | yes | per-residue positional deviation; only when the paired residues are ≥ 50 % identical |

The three Machaon metrics compare per-structure summaries, so they are defined for chains of
different length or sequence — this is what lets Machaon search a whole database for structurally
similar proteins. The other side of it: a value of 0 means *same summary*, not *same structure*, and
residues present in one chain but not the other enter the value. Notebook 03 measures how these
numbers respond to factors other than conformation and sets reference levels; notebook 04 places the
predictions on that scale.